# ADoRE-DoE Phase 2：RSM Design

CCD（ADR-010）→ I-optimal / D-optimal → 約束區域 → augmentation → FDS → seed 管理。
> response 由**合成函數（Synthetic）**產生，僅示範流程；實際使用時請填入真實模擬結果。安裝：`pip install -e ".[plot]"`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adore_doe as ad
from adore_doe import design
os.makedirs('output', exist_ok=True)
SEED = 2026

## 1. 因子

In [ ]:
factors = [ad.Factor("Width", 10, 50, unit="nm"),
           ad.Factor("Spacing", 5, 30, unit="nm"),
           ad.Factor("Thickness", 20, 80, unit="nm")]

## 2. CCD（ADR-010：full factorial、face-centered α=±1、預設 1 個中心點）

In [ ]:
d_ccd = design.ccd(factors, seed=SEED)
print(d_ccd.n_runs, 'runs =', d_ccd.info['n_factorial'], 'factorial +', d_ccd.info['n_axial'], 'axial +', d_ccd.info['n_center'], 'center')
print(d_ccd.info['lof_note'])
d_ccd.actual_df().head(3)

## 3. I-optimal（主要）與 D-optimal（比較），同樣 15 runs

In [ ]:
d_i = design.i_optimal(factors, "quadratic", runs=15, seed=SEED)
d_d = design.d_optimal(factors, "quadratic", runs=15, seed=SEED)
rows = []
for name, d in [("CCD", d_ccd), ("I-optimal", d_i), ("D-optimal", d_d)]:
    rows.append({"design": name, "runs": d.n_runs, "I (avg pred. var, σ²=1)": d.info.get("I_criterion", d.info.get("I_criterion_uniform")),
                 "log det X'X": d.info["logdet_XtX"], "D_eff": d.info["D_efficiency"], "rank": d.info["rank"], "cond(X)": d.info["cond_X"]})
pd.DataFrame(rows).round(4)

In [ ]:
print("I-optimal: starts reaching the best value:", d_i.info["starts_reaching_best"], "of", d_i.info["n_starts"])
print("integration check:", {k: d_i.info["moment_check"][k] for k in ("m_values", "rel_changes", "converged")})

In [ ]:
same = {tuple(r) for r in d_ccd.coded} == {tuple(np.round(r, 9)) for r in d_i.coded}
print("I-optimal (15 runs, k=3) uses exactly the CCD point set:", same)

上面為 True 時代表：在這個條件下，face-centered CCD 本身就是候選集上的 I 最佳設計（所以 FDS 圖中兩條曲線重疊）；這不是錯誤，也不應推廣到其他 k 或 run 數（例如 k=4、25 runs 時 I-optimal 的 I 值優於 CCD）。

starts_reaching_best 很小時，I-optimal 的最佳解只被少數起點找到（啟發式搜尋，不保證全域最佳）；可提高 `n_starts` 並換 seed 重跑確認。

## 4. 線性約束區域（實際單位：Width + Spacing ≤ 60）

In [ ]:
cons = [({"Width": 1, "Spacing": 1}, "<=", 60)]
d_c = design.i_optimal(factors, "quadratic", runs=14, constraints=cons, seed=SEED)
act = d_c.actual_df()
print("max Width+Spacing =", (act.Width + act.Spacing).max(), "(<= 60)")
print("feasible fraction of the cube (acceptance rate):", round(d_c.info["moment"]["acceptance_rate"], 3))

## 5. Augmentation：只建議，不執行模擬

In [ ]:
aug = design.augment(d_i, 3, criterion="I", seed=SEED)
print("I before -> after:", round(aug.value_before, 4), "->", round(aug.value_after, 4))
aug.actual_df()

In [ ]:
# 使用者在這裡自己執行模擬；此處以合成函數代替 (Synthetic)
def synthetic(Z):
    x = np.atleast_2d(Z)
    return 5 + 2*x[:,0] - 1.5*x[:,1] + 0.8*x[:,0]*x[:,1] + 1.2*x[:,2]**2

d_i.add_response("Capacitance", synthetic(d_i.coded))
ids = aug.to_dataset(d_i)                                   # 追加建議點（response 目前是 NaN）
v = d_i.get_response("Capacitance"); v[ids-1] = synthetic(aug.points)
d_i.add_response("Capacitance", v)
print(d_i.n_runs, "runs; sources:", sorted(set(d_i.source)), "; missing:", int(np.isnan(v).sum()))

## 6. FDS plot（多項式模型的預測變異；不是 GPR 變異）

In [ ]:
res = {"CCD": design.fds(d_ccd, seed=1), "I-optimal": design.fds(design.i_optimal(factors, "quadratic", runs=15, seed=SEED), seed=1),
       "D-optimal": design.fds(d_d, seed=1)}
design.plot_fds(res, save_path="output/fds.png"); plt.show()

## 7. Seed 管理

In [ ]:
a = design.i_optimal(factors, "quadratic", runs=15, seed=7, randomize=True)
b = design.i_optimal(factors, "quadratic", runs=15, seed=7, randomize=True)
print("same seed -> identical design and run order:", np.array_equal(a.coded, b.coded) and np.array_equal(a.run_order, b.run_order))
print("recorded seeds:", a.info["seeds"])
u = design.d_optimal(factors, "quadratic", runs=15)           # seed=None: chosen and recorded
print("seed chosen for us:", u.info["seed"], "-> reproducible:", np.array_equal(u.coded, design.d_optimal(factors, "quadratic", runs=15, seed=u.info["seed"]).coded))
sm = ad.core.SeedManager(42); print(sm.child_seed("candidates"), sm.child_seed("moment"))

## 8. 匯出 / 載回（保留設計資訊）

In [ ]:
d_i.to_csv("output/i_optimal_design.csv")
r = ad.ExperimentDataset.from_csv("output/i_optimal_design.csv")
print(r.info["design_type"], r.info["model_spec"]["kind"], r.n_runs, "runs; replicated runs:", r.n_replicated_runs)